In [1]:
import hashlib
import math
import os
from pathlib import Path

import h3pandas
import numpy as np
import pandas as pd
import tensorflow as tf
from tqdm.auto import tqdm

from lib.models.geo_model_net import ResLayer

2025-01-15 13:02:32.800106: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-01-15 13:02:32.800134: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-01-15 13:02:32.801251: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-01-15 13:02:32.806308: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-01-15 13:02:33.346267: W tensorflow/compiler/tf2

In [2]:
def encode_location_elevation(lat, lng, elev):
    norm_lat = np.array([float(lat)]) / 90.0
    norm_lng = np.array([float(lng)]) / 180.0
    norm_loc = tf.stack([norm_lng, norm_lat], axis=1)
    norm_loc = tf.cast(norm_loc, tf.float32)
    elevation = tf.expand_dims(float(elev), axis=0)
    elevation = tf.expand_dims(elevation, axis=0)
    return tf.concat([
        tf.sin(norm_loc * math.pi),
        tf.cos(norm_loc * math.pi),
        elevation,
    ], axis=1)

In [3]:
tax_2_17 = pd.read_csv(
    "/data-ssd/alex/datasets/vision-export-20240929050006-aka-2.17/taxonomy.csv"
)
gp_model_2_17_sinr = tf.keras.models.load_model(
    #"/home/alex/export_eval_models/model_exports/tf_geoprior_2_17_r6_elevation.h5",
    "/home/alex/inatGeoModelTraining/saved_model_2_17_ds_sinr_approach_coords_elev.h5",
    custom_objects={"ResLayer": ResLayer},
    compile=False
)
gp_model_2_17_grid = tf.keras.models.load_model(
    "/home/alex/export_eval_models/model_exports/tf_geoprior_2_17_r6_elevation.h5",
    custom_objects={"ResLayer": ResLayer},
    compile=False
)
elevations = pd.read_csv(
    "/home/alex/elevation_r4_5m.csv",
    index_col="h3_04"
)
above_sea_level_norm = elevations.elevation[elevations.elevation>0] / np.max(elevations.elevation)
below_sea_level_norm = (elevations.elevation[elevations.elevation<0] / np.min(elevations.elevation)) * -1
norm_elev =pd.concat([
    below_sea_level_norm,
    above_sea_level_norm
])
norm_elev_df = pd.DataFrame({
    "elevation": norm_elev
})


2025-01-15 13:02:33.935901: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:274] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2025-01-15 13:02:33.935925: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:129] retrieving CUDA diagnostic information for host: euclid
2025-01-15 13:02:33.935929: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:136] hostname: euclid
2025-01-15 13:02:33.935995: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:159] libcuda reported version is: 550.120.0
2025-01-15 13:02:33.936010: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:163] kernel reported version is: 550.120.0
2025-01-15 13:02:33.936018: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:241] kernel version seems to match DSO: 550.120.0


In [4]:
spatial_data = pd.read_parquet(
    "/home/alex/sinr/data/train/geo_prior_train.parquet",
    columns=[
        "longitude",
        "latitude",
        "taxon_id",
    ]
)
num_obs = len(spatial_data)
spatial_data = spatial_data[
    (
        (spatial_data["latitude"] <= 90)
        & (spatial_data["latitude"] >= -90)
        & (spatial_data["longitude"] <= 180)
        & (spatial_data["longitude"] >= -180)
    )
]
if (num_obs - len(spatial_data)) > 0:
    print(f"  {num_obs - len(spatial_data)} items filtered due to invalid locs")

num_obs = len(spatial_data)
spatial_data = spatial_data.dropna()
if (num_obs - len(spatial_data)) > 0:
    print(f"  {num_obs - len(spatial_data)} items filtered due to NaN entry")

print(f"  after cleaning, we have {len(spatial_data)} records.")

taxon_ids = spatial_data["taxon_id"].values.astype(int)
unique_taxa, class_ids = np.unique(taxon_ids, return_inverse=True)

  after cleaning, we have 35500262 records.


In [5]:
gp_eval_dir = Path("/home/alex/sinr/data/eval/geo_prior")
sinr_gp_model_meta = pd.read_csv(
    gp_eval_dir / "geo_prior_model_meta.csv"
)
sinr_gp_model_meta.reset_index(inplace=True)
gp_model_preds = np.load(
    gp_eval_dir / "geo_prior_model_preds.npz"
)

tax_in_gp_task = set(sinr_gp_model_meta.taxon_id.unique())

leaf_tax_2_17 = tax_2_17[~tax_2_17.leaf_class_id.isna()]
tax_in_gp_model = set(leaf_tax_2_17.taxon_id.unique())

tax_in_both_models = tax_in_gp_task.intersection(tax_in_gp_model)


In [6]:
gp_model_preds["labels"].max()

52496

In [7]:
# sinr_gp_model_meta = sinr_gp_model_meta[sinr_gp_model_meta.taxon_id.isin(tax_in_both_models)]

In [8]:
sinr_gp_model_meta[sinr_gp_model_meta.index==6007]

,index,photo_uuid,photo_id,observation_uuid,observer_id,taxon_id,leaf_class_id,latitude,longitude,positional_accuracy,observed_on
6007,6007,3b923216-c6c7-46ff-aa6a-97995be58801,174944622,3cedc8b1-381a-4741-8f7c-24ca8c9209b9,2226520,559298,13938,-28.660912,-70.918604,NaN,2022-01-04


In [9]:
sinr_gp_model_meta_dfh3 = sinr_gp_model_meta.h3.geo_to_h3(
    resolution=4,
    lat_col="latitude",
    lng_col="longitude",
)
sinr_gp_model_meta_elev = pd.merge(
    sinr_gp_model_meta_dfh3,
    norm_elev_df,
    how="inner",
    left_index=True,
    right_index=True
)


In [10]:
sinr_gp_model_meta_dfh3[sinr_gp_model_meta_dfh3["index"]==6007]

,index,photo_uuid,photo_id,observation_uuid,observer_id,taxon_id,leaf_class_id,latitude,longitude,positional_accuracy,observed_on
h3_04,,,,,,,,,,,
84b21d9ffffffff,6007,3b923216-c6c7-46ff-aa6a-97995be58801,174944622,3cedc8b1-381a-4741-8f7c-24ca8c9209b9,2226520,559298,13938,-28.660912,-70.918604,NaN,2022-01-04


In [11]:
norm_elev_df

,elevation
h3_04,
8400001ffffffff,-1.000000
8400003ffffffff,-1.000000
8400005ffffffff,-1.000000
8400007ffffffff,-1.000000
8400009ffffffff,-1.000000
...,...
84f3b65ffffffff,0.343705
84f3b67ffffffff,0.338236
84f3b69ffffffff,0.323223


In [12]:
sinr_gp_model_meta_elev.reset_index(inplace=True)
sinr_gp_model_meta_elev.set_index("index", inplace=True)


In [13]:
sinr_gp_model_meta_elev[sinr_gp_model_meta_elev["h3_04"] == "84b21d9ffffffff"]

,h3_04,photo_uuid,photo_id,observation_uuid,observer_id,taxon_id,leaf_class_id,latitude,longitude,positional_accuracy,observed_on,elevation
index,,,,,,,,,,,,


In [14]:
all_probs = gp_model_preds["probs"]
all_inds = gp_model_preds["inds"]
all_taxon_ids = gp_model_preds["model_to_taxa"]

leaf_tax_2_17 = tax_2_17[~tax_2_17.leaf_class_id.isna()]
leaf_tax_2_17 = leaf_tax_2_17.sort_values(
    "spatial_class_id"
)
leaf_tax_2_17.set_index(
    "spatial_class_id",
    inplace=True
)

In [15]:
i = np.random.randint(100_000)

In [16]:
row = sinr_gp_model_meta_elev.iloc[i]
print(row)

probs = all_probs[i]
inds = all_inds[i]
taxon_ids = all_taxon_ids[inds]

h3_04                                       84da8c7ffffffff
photo_uuid             e2220c8c-bfe7-440b-8d22-b42af4f5459e
photo_id                                          176642645
observation_uuid       0deeb0ed-1344-4c7c-b501-947ed4feb3de
observer_id                                           57756
taxon_id                                             323375
leaf_class_id                                          1980
latitude                                         -45.707314
longitude                                        166.555236
positional_accuracy                                   143.0
observed_on                                      2022-01-11
elevation                                         -0.314867
Name: 33498, dtype: object


In [17]:
inds[-1], all_taxon_ids[inds[-1]]

(30034, 33535)

In [18]:
np.where(all_taxon_ids==row.taxon_id)

(array([1980]),)

In [19]:
np.where(inds==9191)

(array([], dtype=int64),)

# let's try to match what they're doing in the repo

In [20]:
gp_model_preds

NpzFile '/home/alex/sinr/data/eval/geo_prior/geo_prior_model_preds.npz' with keys: labels, probs, inds, photo_uuid, model_to_taxa

In [21]:
correct_pred = np.zeros(
    gp_model_preds["probs"].shape[0]
)

In [22]:
vision_probs = gp_model_preds["probs"][0:,]
vision_inds = gp_model_preds["inds"][0:,]
gt = gp_model_preds["labels"][0]

In [23]:
vision_pred = np.zeros((1, leaf_tax_2_17.shape[0]), dtype=np.float32)
vision_pred[np.arange(vision_pred.shape[0])[...,np.newaxis], vision_inds] = vision_probs
vision_pred

array([[1.5828608e-07, 4.5639983e-05, 5.1950298e-08, ..., 0.0000000e+00,
        0.0000000e+00, 0.0000000e+00]], dtype=float32)

In [24]:
float((gp_model_preds['inds'][:, -1] == gp_model_preds['labels']).mean())

0.7540940157046231

In [25]:
all_inds = gp_model_preds["inds"]
all_labels = gp_model_preds["labels"]
labels_to_tax = gp_model_preds["model_to_taxa"]

seen = 0
correct_labels = 0
correct_tids = 0

for i in tqdm(range(len(all_inds))):
    row = sinr_gp_model_meta.loc[i]
    top_ind = all_inds[i, -1]
    top_label = all_labels[i]
    taxon_id = labels_to_tax[top_ind]

    if top_ind == top_label:
        correct_labels += 1
    if taxon_id == row["taxon_id"]:
        correct_tids += 1

    

    seen += 1

print(correct_labels / seen)
print(correct_tids / seen)

  0%|          | 0/282974 [00:00<?, ?it/s]

0.7540940157046231
0.7540940157046231


In [26]:
i = 50_000
vision_probs = all_probs[i]

vision_scores = pd.DataFrame({
    "probs": all_probs[i],
    "labels": all_inds[i],
    "taxon_id": labels_to_tax[all_inds[i]],
})
vision_scores, all_labels[i] ,sinr_gp_model_meta.loc[i].taxon_id

(            probs  labels  taxon_id
 0    9.958450e-08   55363    474089
 1    9.958915e-08   28234    148074
 2    9.967648e-08    2899    130177
 3    9.984868e-08   44643     26074
 4    1.003773e-07   18821    124542
 ..            ...     ...       ...
 507  2.590470e-02   41585      3855
 508  6.940381e-02   39606      3838
 509  7.300100e-02   17788     72458
 510  1.278017e-01    9707      3978
 511  5.195680e-01    6268      3837
 
 [512 rows x 3 columns],
 6268,
 3837)

In [27]:
loc_input = encode_location_elevation(
    sinr_gp_model_meta_elev.loc[i].latitude,
    sinr_gp_model_meta_elev.loc[i].longitude,
    sinr_gp_model_meta_elev.loc[i].elevation
)

In [29]:
gp_preds = gp_model_2_17_grid.predict(loc_input)[0]

1/1 [==============================] - 0s 61ms/step


In [30]:
geo_scores = pd.DataFrame({
    "gp_preds": gp_preds,
    "taxon_id": leaf_tax_2_17.taxon_id
})

In [31]:
merged_scores = pd.merge(
    vision_scores,
    geo_scores
)

In [32]:
merged_scores["comb_scores"] = merged_scores["probs"] * merged_scores["gp_preds"]

In [33]:
merged_scores.sort_values("comb_scores").iloc[-1].taxon_id

3837.0

In [38]:
seen = 0
correct_vision = 0
correct_comb_grid = 0
correct_comb_sinr = 0

for i in tqdm(range(len(all_inds))):
    try:
        row = sinr_gp_model_meta_elev.loc[i]
    except:
        continue
    top_ind = all_inds[i, -1]
    taxon_id = labels_to_tax[top_ind]

    if taxon_id == row["taxon_id"]:
        correct_vision += 1


    vision_scores = pd.DataFrame({
        "probs": all_probs[i],
        "labels": all_inds[i],
        "taxon_id": labels_to_tax[all_inds[i]],
    })
    vision_scores, all_labels[i] ,sinr_gp_model_meta.loc[i].taxon_id
    try:
        loc_input = encode_location_elevation(
            sinr_gp_model_meta_elev.loc[i].latitude,
            sinr_gp_model_meta_elev.loc[i].longitude,
            sinr_gp_model_meta_elev.loc[i].elevation
        )
    except:
        continue
    
    gp_preds_grid = gp_model_2_17_grid.predict(loc_input, verbose=0)[0]
    geo_scores = pd.DataFrame({
        "gp_preds": gp_preds_grid,
        "taxon_id": leaf_tax_2_17.taxon_id
    })
    merged_scores = pd.merge(
        vision_scores,
        geo_scores
    )
    merged_scores["comb_scores"] = merged_scores["probs"] * merged_scores["gp_preds"]
    comb_tid = merged_scores.sort_values("comb_scores").iloc[-1].taxon_id
    if comb_tid == row["taxon_id"]:
        correct_comb_grid += 1

    gp_preds_sinr = gp_model_2_17_sinr.predict(loc_input, verbose=0)[0]
    geo_scores = pd.DataFrame({
        "gp_preds": gp_preds_sinr,
        "taxon_id": leaf_tax_2_17.taxon_id
    })
    merged_scores = pd.merge(
        vision_scores,
        geo_scores
    )
    merged_scores["comb_scores"] = merged_scores["probs"] * merged_scores["gp_preds"]
    comb_tid = merged_scores.sort_values("comb_scores").iloc[-1].taxon_id
    if comb_tid == row["taxon_id"]:
        correct_comb_sinr += 1

    seen += 1
    


  0%|          | 0/282974 [00:00<?, ?it/s]

In [39]:
print(f"vision correct    : {correct_vision/seen*100:.2f}")
print(f"comb correct grid : {correct_comb_grid/seen*100:.2f}")
print(f"comb correct sinr : {correct_comb_sinr/seen*100:.2f}")

vision correct    : 75.41
comb correct grid : 82.94
comb correct sinr : 83.61


In [43]:
print(f"grid improvement : {correct_comb_grid/seen*100 - correct_vision/seen*100:.2f}")
print(f"sinr improvement : {correct_comb_sinr/seen*100 - correct_vision/seen*100:.2f}")



grid improvement : 7.53
sinr improvement : 8.20


In [ ]:
82.6-75.9

In [ ]:
84-75.9